# Project 05: Quantum RL with Noise and Mitigation

This notebook provides a starter template for policy-gradient training with a parameterized quantum policy.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

## 1) Define Toy Environment

Replace this with Gymnasium if needed.

In [ ]:
class TwoStateEnv:
    def __init__(self):
        self.state = 0

    def reset(self):
        self.state = np.random.choice([0, 1])
        return self.state

    def step(self, action):
        reward = 1.0 if action == self.state else -1.0
        done = True
        return self.state, reward, done, {}

env = TwoStateEnv()

## 2) Quantum Policy Circuit

In [ ]:
def build_policy_circuit(state, theta):
    qc = QuantumCircuit(1, 1)
    if state == 1:
        qc.x(0)
    qc.ry(theta, 0)
    qc.measure(0, 0)
    return qc

def sample_action(state, theta, simulator, shots=256):
    qc = build_policy_circuit(state, theta)
    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()
    p1 = counts.get("1", 0) / shots
    return 1 if np.random.rand() < p1 else 0

## 3) Noise Model + Simulators

In [ ]:
clean_sim = AerSimulator()

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.02, 1), ["ry", "x"])
readout = ReadoutError([[0.97, 0.03], [0.04, 0.96]])
noise_model.add_all_qubit_readout_error(readout)

noisy_sim = AerSimulator(noise_model=noise_model)

## 4) Policy Gradient Training Loop (Starter)

This uses finite-difference gradient for simplicity.

In [ ]:
def run_episode(theta, simulator, shots=256):
    s = env.reset()
    a = sample_action(s, theta, simulator, shots=shots)
    _, r, _, _ = env.step(a)
    return r

def train(simulator, episodes=200, lr=0.1, eps=0.1):
    theta = 0.1
    rewards = []
    for _ in range(episodes):
        r = run_episode(theta, simulator)
        r_plus = run_episode(theta + eps, simulator)
        r_minus = run_episode(theta - eps, simulator)
        grad = (r_plus - r_minus) / (2 * eps)
        theta += lr * grad
        rewards.append(r)
    return theta, rewards

theta_clean, rewards_clean = train(clean_sim)
theta_noisy, rewards_noisy = train(noisy_sim)

## 5) Simple Mitigation and Comparison

In [ ]:
# Placeholder mitigation: increase shots to reduce sampling noise.
# Replace with richer methods (e.g., measurement calibration) later.
def run_episode_mitigated(theta, simulator, shots=2048):
    s = env.reset()
    a = sample_action(s, theta, simulator, shots=shots)
    _, r, _, _ = env.step(a)
    return r

def train_mitigated(simulator, episodes=200, lr=0.1, eps=0.1):
    theta = 0.1
    rewards = []
    for _ in range(episodes):
        r = run_episode_mitigated(theta, simulator)
        r_plus = run_episode_mitigated(theta + eps, simulator)
        r_minus = run_episode_mitigated(theta - eps, simulator)
        grad = (r_plus - r_minus) / (2 * eps)
        theta += lr * grad
        rewards.append(r)
    return theta, rewards

theta_mit, rewards_mit = train_mitigated(noisy_sim)

plt.figure(figsize=(9, 4))
plt.plot(rewards_clean, label="Clean")
plt.plot(rewards_noisy, label="Noisy")
plt.plot(rewards_mit, label="Noisy + Mitigation")
plt.title("Quantum RL Reward Curves")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(alpha=0.3)
plt.show()